[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lmassaron/finetuning/blob/main/chapter_08/listing_8.1.ipynb)

In [1]:
import sys
if 'google.colab' in sys.modules:
    pass

### Listing 6.25: Library Imports and Hardware Precision Setup

In [ ]:
import os
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
import torch
from datasets import load_dataset

from trl import SFTTrainer, SFTConfig

compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
    else torch.float16
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


### Listing 6.26: Loading the Vision-Language Model and Configuring LoRA Adapters

In [3]:
MODEL_ID = "unsloth/Qwen2-VL-2B-Instruct"

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=MODEL_ID,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=3407,
)

==((====))==  Unsloth 2026.7.6: Fast Qwen2_Vl patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Skipping model.language_model.layers.1.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.down_proj: no quant_state found


### Listing 6.28: Configuring SFTTrainer and Fine-Tuning Qwen2-VL

In [4]:
dataset = load_dataset("unsloth/LaTeX_OCR", split="train")

shuffled_dataset = dataset.shuffle(seed=42)
train_ds = shuffled_dataset.select(range(500))
eval_ds = shuffled_dataset.select(range(500, 550))


def convert_to_conversation(sample):
    conversation = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Write the LaTeX representation for this image.",
                },
                {"type": "image"},
            ],
        },
        {"role": "assistant", "content": [{"type": "text", "text": sample["text"]}]},
    ]
    return {"messages": conversation, "images": [sample["image"]]}


train_mapped = train_ds.map(
    convert_to_conversation, remove_columns=train_ds.column_names
)
eval_mapped = eval_ds.map(convert_to_conversation, remove_columns=eval_ds.column_names)

### Listing 6.28: Configuring SFTTrainer and Fine-Tuning Qwen2-VL

In [5]:
training_args = SFTConfig(
    output_dir="qwen2-vl-latex",
    dataset_text_field="text",
    max_seq_length=512,
    remove_unused_columns=False,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=0,
    max_steps=30,
    bf16=(compute_dtype == torch.bfloat16),
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_mapped,
    eval_dataset=eval_mapped,
    args=training_args,
)

model.config.use_cache = False

trainer.train()

model.save_pretrained("qwen2-vl-latex-adapter")
tokenizer.save_pretrained("qwen2-vl-latex-adapter")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None}.


Unsloth: Model does not have a default image size - using 512


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 28,950,528 of 2,237,936,128 (1.29% trained)


Step,Training Loss,Validation Loss
30,No log,0.151498


Unsloth: Restored added_tokens_decoder metadata in qwen2-vl-latex/checkpoint-30/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen2-vl-latex-adapter/tokenizer_config.json.


['qwen2-vl-latex-adapter/processor_config.json']

### Listing 6.29: Running Inference and Evaluating the Fine-Tuned Vision Model

In [6]:
FastVisionModel.for_inference(model)
model.generation_config.max_length = None

sample = eval_ds[0]
image = sample["image"]
expected_latex = sample["text"]

messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Write the LaTeX representation for this image."},
            {"type": "image", "image": image},
        ],
    }
]

input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(image, input_text, add_special_tokens=False, return_tensors="pt").to(
    "cuda"
)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=128)

generated_text = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[1] :], skip_special_tokens=True
).strip()

print("--- Expected LaTeX ---")
print(expected_latex)
print("\n--- Generated LaTeX ---")
print(generated_text)

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Expected LaTeX ---
\Gamma _ { \sigma } + \Gamma _ { m } = \int d ^ { 2 } x [ - \frac { 1 } { 8 \pi } T r ( \partial _ { \mu } U \partial _ { \mu } U ^ { \dag } ) + \frac { 1 } { 2 } m ^ { 2 } T r ( U + U ^ { \dag } - 2 ) ] ,

--- Generated LaTeX ---
\Gamma _ { \sigma } + \Gamma _ { m } = \int d ^ { 2 } x [ - \frac { 1 } { 8 \pi } T r ( \partial _ { \mu } U \partial _ { \mu } U ^ { \dagger } ) + \frac { 1 } { 2 } m ^ { 2 } T r ( U + U ^ { \dagger } - 2 ) ] ,
